<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/08-random-forests.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 8 — Random Forests and Bagging

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/random-forests-and-bagging/).

Variance reduction through disagreement: many trees, each slightly wrong in its own way, voting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

def make_loans(n, seed=0):
    rng = np.random.default_rng(seed)
    income = rng.uniform(15, 120, n)
    debt_ratio = rng.uniform(0.0, 0.8, n)
    missed = rng.integers(0, 4, n)
    age = rng.uniform(21, 70, n)
    late_fees = 12 * missed + rng.normal(0, 8, n)
    noise = rng.normal(size=(n, 5))                                   # five columns of nothing
    risky = ((income < 40) & (debt_ratio > 0.4)) | (missed >= 2)
    default = np.where(risky, rng.uniform(size=n) < 0.7, rng.uniform(size=n) < 0.05).astype(int)
    return np.column_stack([income, debt_ratio, missed, age, late_fees, noise]), default

names = ['income', 'debt_ratio', 'missed', 'age', 'late_fees'] + [f'noise{i}' for i in range(5)]
X, y = make_loans(600, seed=1)
X_test, y_test = make_loans(5000, seed=2)
print(f'{len(y)} applications, {X.shape[1]} features, {y.mean():.0%} defaulted')

## 1. The arithmetic of averaging

In [ ]:
rng = np.random.default_rng(0)
sigma, B = 1.0, 25
for rho in [0.0, 0.3, 0.7, 1.0]:
    # B estimates that share a common component (correlation rho) and have their own noise
    shared = rng.normal(0, 1, 10000)
    own = rng.normal(0, 1, (10000, B))
    estimates = np.sqrt(rho) * shared[:, None] + np.sqrt(1 - rho) * own
    averaged = estimates.mean(axis=1)
    formula = rho * sigma**2 + (1 - rho) * sigma**2 / B
    print(f'correlation {rho:.1f}: variance of one estimate {estimates.var():.3f}, of the average of {B}: '
          f'{averaged.var():.3f}  (formula says {formula:.3f})')

## 2. Bootstrap samples, and the rows left out

In [ ]:
rng = np.random.default_rng(1)
n = len(y)
in_bag = np.zeros(10)
for t in range(10):
    sample = rng.choice(n, n, replace=True)
    in_bag[t] = len(np.unique(sample)) / n
print(f'each bootstrap sample uses about {in_bag.mean():.1%} of the distinct rows; the rest are "out of bag"')
print(f'theory: 1 - 1/e = {1 - np.exp(-1):.1%}')

## 3. Bagging, from scratch

In [ ]:
def bagged_trees(X, y, n_trees, max_features=None, seed=0):
    """Fully grown trees, each on a bootstrap sample. max_features limits the features
    each SPLIT may choose from, which is the one change that turns bagging into a random forest."""
    rng = np.random.default_rng(seed)
    trees, oob_votes, oob_counts = [], np.zeros(len(y)), np.zeros(len(y))
    for t in range(n_trees):
        sample = rng.choice(len(y), len(y), replace=True)
        tree = DecisionTreeClassifier(max_features=max_features, random_state=t).fit(X[sample], y[sample])
        trees.append(tree)
        out = np.setdiff1d(np.arange(len(y)), sample)             # rows this tree never saw
        oob_votes[out] += tree.predict(X[out])
        oob_counts[out] += 1
    seen = oob_counts > 0
    oob_accuracy = np.mean((oob_votes[seen] / oob_counts[seen] >= 0.5) == y[seen])
    return trees, oob_accuracy

def forest_predict(trees, X):
    votes = np.mean([t.predict(X) for t in trees], axis=0)
    return (votes >= 0.5).astype(int)

single = DecisionTreeClassifier(random_state=0).fit(X, y)
print(f'one fully grown tree                      : test accuracy {single.score(X_test, y_test):.3f}')
trees, oob = bagged_trees(X, y, 100)
print(f'100 bagged trees, every feature at a split: test accuracy {np.mean(forest_predict(trees, X_test) == y_test):.3f}  (out-of-bag estimate {oob:.3f})')
trees, oob = bagged_trees(X, y, 100, max_features=3)
print(f'100 trees, 3 random features per split    : test accuracy {np.mean(forest_predict(trees, X_test) == y_test):.3f}  (out-of-bag estimate {oob:.3f})')

## 4. Why the random subset of features helps

In [ ]:
def tree_correlation(trees, X):
    preds = np.array([t.predict(X) for t in trees], dtype=float)
    c = np.corrcoef(preds)
    return c[np.triu_indices_from(c, 1)].mean()

print(f'{"max_features":>12} {"tree accuracy":>14} {"correlation":>12} {"forest accuracy":>16}')
for mf in [None, 5, 3, 1]:
    trees, _ = bagged_trees(X, y, 50, max_features=mf, seed=3)
    one = np.mean([t.score(X_test, y_test) for t in trees])
    acc = np.mean(forest_predict(trees, X_test) == y_test)
    print(f'{str(mf):>12} {one:>14.3f} {tree_correlation(trees, X_test):>12.3f} {acc:>16.3f}')

## 5. The one line you would ship

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0, n_jobs=-1).fit(X, y)
print(f'RandomForestClassifier: test accuracy {rf.score(X_test, y_test):.3f}, out-of-bag estimate {rf.oob_score_:.3f}')

## 6. More trees never hurt, and stop helping

In [ ]:
print(f'{"trees":>6} {"test":>7}')
for n_trees in [1, 5, 25, 100, 300, 1000]:
    f = RandomForestClassifier(n_estimators=n_trees, random_state=0, n_jobs=-1).fit(X, y)
    print(f'{n_trees:>6} {f.score(X_test, y_test):>7.3f}')

## 7. Timid probabilities

In [ ]:
p_rf = rf.predict_proba(X_test)[:, 1]
print(f'random forest probabilities: {np.mean(p_rf < 0.05):.1%} below 0.05, {np.mean(p_rf > 0.95):.1%} above 0.95')
bins = np.linspace(0, 1, 6)
which = np.digitize(p_rf, bins) - 1
print(f'{"predicted":>12} {"actual rate":>12} {"count":>6}')
for b in range(5):
    m = which == b
    if m.sum() > 50:
        print(f'{bins[b]:>5.1f}-{bins[b + 1]:<5.1f} {y_test[m].mean():>12.2f} {m.sum():>6}')

## 8. Which features matter, two ways

In [ ]:
print(f'{"feature":<11} {"impurity":>9} {"permutation":>12}')
rng = np.random.default_rng(5)
base_acc = rf.score(X_test, y_test)
for j, name in enumerate(names):
    shuffled = X_test.copy()
    shuffled[:, j] = rng.permutation(shuffled[:, j])             # break the link between this feature and the label
    drop = base_acc - rf.score(shuffled, y_test)
    print(f'{name:<11} {rf.feature_importances_[j]:>9.3f} {drop:>12.3f}')

## 9. Forests predict numbers too, and still cannot extrapolate

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def bicycle(km):
    return 10 + 3.2 * km + 0.15 * km**2

rng = np.random.default_rng(1)
km = rng.uniform(0.5, 8.0, 200); minutes = bicycle(km) + rng.normal(0, 4.0, 200)
km_t = rng.uniform(0.5, 8.0, 1000); min_t = bicycle(km_t) + rng.normal(0, 4.0, 1000)

tree_r = DecisionTreeRegressor(random_state=0).fit(km.reshape(-1, 1), minutes)
rf_r = RandomForestRegressor(n_estimators=300, random_state=0).fit(km.reshape(-1, 1), minutes)
for name, m in [('one deep tree', tree_r), ('300 trees', rf_r)]:
    print(f'{name:<14}: {np.mean(np.abs(m.predict(km_t.reshape(-1, 1)) - min_t)):.2f} min on new orders')
print(f'the true formula: {np.mean(np.abs(bicycle(km_t) - min_t)):.2f} min')
print(f'at 16 km: truth {bicycle(16):.1f}, forest {rf_r.predict([[16]])[0]:.1f}')

## 10. What a forest costs at prediction time

In [ ]:
import time
from sklearn.linear_model import LogisticRegression

batch = X_test[:1000]
for name, m in [('logistic regression', LogisticRegression(max_iter=1000).fit(X, y)),
                ('one tree', single),
                ('forest, 300 trees', rf)]:
    start = time.perf_counter()
    for _ in range(20):
        m.predict(batch)
    ms = (time.perf_counter() - start) / 20 * 1000
    print(f'{name:<20} {ms:7.2f} ms per 1,000 applications')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
grid = np.linspace(0.5, 8, 400).reshape(-1, 1)
ax.scatter(km, minutes, s=10, color='grey', alpha=0.6)
ax.plot(grid, tree_r.predict(grid), lw=1, alpha=0.7, label='one deep tree')
ax.plot(grid, rf_r.predict(grid), lw=2, label='300 trees averaged')
ax.plot(grid, bicycle(grid), color='black', lw=1, ls='--', label='truth')
ax.set_xlabel('distance (km)'); ax.set_ylabel('minutes'); ax.legend()
plt.show()

## Try it yourself

1. In section 3, grow the bagged trees with `max_depth=3`. Does bagging still help? Why less?
2. Set `max_features=1` in `RandomForestClassifier`. With five noise features, what happens to accuracy, and what does the correlation table in section 4 predict?
3. In section 8, double the noise on `late_fees`. Which importance method notices first?